<a href="https://colab.research.google.com/github/treborskrub/Trinary-Core-Processor-Architecture-T-CPU-/blob/main/trinary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
from enum import IntEnum

class TrinaryLogic(IntEnum):
    RELEVANT = 1      # Wave Crest (+1)   -> Active Value
    UNKNOWN = 0       # Ground Neutral (0) -> Structural Data Absence
    IRRELEVANT = -1   # Wave Trough (-1)   -> Path Void / Early Exit Trigger

class TrinaryProcessor:
    """
    Emulates a 14-node Balanced Ternary Processor.
    Operates natively on vectors of (-1, 0, +1) using wave-domain operations.
    """
    def __init__(self, num_nodes: int = 14):
        self.num_nodes = num_nodes
        # Registers: Each holds a 14-node ternary wavefront array
        self.registers = {
            "R1": np.zeros(self.num_nodes),  # General Purpose 1
            "R2": np.zeros(self.num_nodes),  # General Purpose 2
            "ACC": np.zeros(self.num_nodes), # Wavefront Accumulator
            "SR": 0                          # Status Register (Global State Context)
        }
        # Sieve Hardware Integration for pipeline security
        self.phi = 1.61803398875
        self.sieve_threshold = np.pi * self.phi * (1.0 / 3.0)
        self.lambda_max = 1.35

    def load_register(self, reg_name: str, wave_vector: np.ndarray):
        """Loads an input wavefront directly into the processor registers."""
        if len(wave_vector) != self.num_nodes:
            raise ValueError(f"Vector must precisely match the {self.num_nodes}-node fabric grid.")
        self.registers[reg_name] = np.copy(wave_vector)

    def execute_and(self, src_reg1: str, src_reg2: str, dest_reg: str):
        """
        Trinary AND (Strict Filter Fallback Logic):
        Applies element-wise min(). Irrelevant (-1) wins over everything.
        """
        v1 = self.registers[src_reg1]
        v2 = self.registers[src_reg2]
        self.registers[dest_reg] = np.minimum(v1, v2)
        self._update_status_register(dest_reg)

    def execute_or(self, src_reg1: str, src_reg2: str, dest_reg: str):
        """
        Trinary OR (Relevance Priority Logic):
        Applies element-wise max(). Relevant (+1) wins over everything.
        """
        v1 = self.registers[src_reg1]
        v2 = self.registers[src_reg2]
        self.registers[dest_reg] = np.maximum(v1, v2)
        self._update_status_register(dest_reg)

    def execute_refract_rotation(self, src_reg: str, dest_reg: str):
        """
        Executes a hardware-level spatial refraction cycle.
        Applies the Pi-Phi-1/3 arrangement loop.
        """
        state = np.copy(self.registers[src_reg])
        rotation_matrix = np.roll(state, 1) * (1.0 / 3.0)

        # Wavefront propagation
        new_state = (state * np.cos(self.sieve_threshold)) + (rotation_matrix * np.sin(self.sieve_threshold))

        # Enforce Shortfall Constraints: Snap back to ternary limits to avoid floating overflow
        self.registers[dest_reg] = np.clip(np.round(new_state), -1.0, 1.0)
        self._update_status_register(dest_reg)

    def _update_status_register(self, monitored_reg: str):
        """
        Updates the processor status using Early-Exit Logic rules.
        If a register drops entirely into the VOID state (-1), status sets a hardware halt.
        """
        target_vector = self.registers[monitored_reg]
        if np.min(target_vector) == TrinaryLogic.IRRELEVANT and np.max(target_vector) <= 0.0:
            self.registers["SR"] = int(TrinaryLogic.IRRELEVANT)  # VOIDED BRANCH SIGNIFIED
        elif np.all(target_vector == 0.0):
            self.registers["SR"] = int(TrinaryLogic.UNKNOWN)     # UNMAPPED STANDBY SIGNIFIED
        else:
            self.registers["SR"] = int(TrinaryLogic.RELEVANT)    # ACTIVE SYSTEM HEALTHY

# Hardware Sandbox Verification
if __name__ == "__main__":
    t_cpu = TrinaryProcessor()

    # Simulate a stream package coming into R1
    package_r1 = np.array([1., 0., 1., 1., 0., -1., 0., 1., -1., 0., 0., 1., 1., 0.])
    # Simulate a masking filter in R2
    mask_r2    = np.array([1., 1., 1., 1., 1., -1., 1., 1., -1., 1., 1., 1., 1., 1.])

    t_cpu.load_register("R1", package_r1)
    t_cpu.load_register("R2", mask_r2)

    print("Executing Strict AND Filtering into Accumulator...")
    t_cpu.execute_and("R1", "R2", "ACC")
    print(f"Accumulator Vector  : {t_cpu.registers['ACC']}")
    print(f"Hardware Status Code: {t_cpu.registers['SR']} (Status: {TrinaryLogic(t_cpu.registers['SR']).name})")

Executing Strict AND Filtering into Accumulator...
Accumulator Vector  : [ 1.  0.  1.  1.  0. -1.  0.  1. -1.  0.  0.  1.  1.  0.]
Hardware Status Code: 1 (Status: RELEVANT)


In [3]:

import numpy as np
import time
from typing import Dict, List, Callable

# Ensure our logical types are completely integrated in the cell scope
class TrinaryLogic:
    RELEVANT = 1      # Wave Crest (+1)   -> Active Value
    UNKNOWN = 0       # Ground Neutral (0) -> Structural Data Absence
    IRRELEVANT = -1   # Wave Trough (-1)   -> Path Void / Early Exit Trigger

class TrinaryProcess:
    """
    Represents a native task running on TrinaryOS.
    Every process manages its own 14-node wave signature block.
    """
    def __init__(self, pid: int, name: str, execution_vector: np.ndarray, ability_callback: Callable):
        self.pid = pid
        self.name = name
        self.vector = np.copy(execution_vector)
        self.ability_callback = ability_callback
        self.state = TrinaryLogic.UNKNOWN
        self.runtime_cycles = 0

class TrinaryOS:
    """
    The core operating system kernel. Natively orchestrates processes
    by screening their wavefront signatures using Early-Exit rules.
    """
    def __init__(self, hardware_cpu: TrinaryProcessor):
        self.cpu = hardware_cpu
        self.process_table: Dict[int, TrinaryProcess] = {}
        self.next_pid = 100

        # Core OS Queues mapped to your trinary states
        self.active_queue: List[int] = []     # State: +1
        self.shortfall_queue: List[int] = []  # State:  0
        self.void_queue: List[int] = []       # State: -1

    def register_ability(self, name: str, wavefront_signature: np.ndarray, callback: Callable) -> int:
        """Loads one of your 12 core abilities into the OS process table."""
        pid = self.next_pid
        process = TrinaryProcess(pid, name, wavefront_signature, callback)
        self.process_table[pid] = process
        self.next_pid += 1

        # Triage immediately on registration using the hardware rules
        self._triage_process(process)
        return pid

    def _triage_process(self, process: TrinaryProcess):
        """OS Kernel Triage: Automatically segments processes based on their wave signature."""
        min_state = np.min(process.vector)
        max_state = np.max(process.vector)

        # Early-Exit Check: If the entire block is unmapped noise/void, kill it before scheduling
        if min_state == TrinaryLogic.IRRELEVANT and max_state <= 0.0:
            process.state = TrinaryLogic.IRRELEVANT
            if process.pid not in self.void_queue:
                self.void_queue.append(process.pid)
            print(f"KERNEL: Process '{process.name}' [PID {process.pid}] flagged as IRRELEVANT. Routed to Void Queue.")

        # Shortfall Standby Check: Has data gaps but holds structural potential
        elif np.all(process.vector == 0.0) or (min_state == 0.0 and max_state == 0.0):
            process.state = TrinaryLogic.UNKNOWN
            if process.pid not in self.shortfall_queue:
                self.shortfall_queue.append(process.pid)
            print(f"KERNEL: Process '{process.name}' [PID {process.pid}] flagged as UNKNOWN. Parked in Shortfall Standby.")

        # Active Payload Check
        else:
            process.state = TrinaryLogic.RELEVANT
            if process.pid not in self.active_queue:
                self.active_queue.append(process.pid)
            print(f"KERNEL: Process '{process.name}' [PID {process.pid}] flagged as RELEVANT. Scheduled to Active Queue.")

    def run_scheduler_cycle(self):
        """Executes a single clock tick across the entire OS environment."""
        print(f"\n--- OS Scheduler Cycle Execution ---")
        print(f"Active Tasks: {len(self.active_queue)} | Shortfall Standby: {len(self.shortfall_queue)} | Void Cleaned: {len(self.void_queue)}")

        # 1. Instantly purge the void queue to keep system fabric completely clear
        if self.void_queue:
            print(f"  [Purge Engine] Garbage collecting {len(self.void_queue)} dead-end threads to preserve registers.")
            for pid in list(self.void_queue):
                del self.process_table[pid]
                self.void_queue.remove(pid)

        # 2. Execute active tasks through the virtual T-CPU registers
        for pid in list(self.active_queue):
            proc = self.process_table[pid]

            # Load task wave directly into the hardware execution context
            self.cpu.load_register("R1", proc.vector)

            # Trigger hardware spatial refraction to cycle the wave signature
            self.cpu.execute_refract_rotation("R1", "ACC")

            # Execute the actual capability logic
            proc.ability_callback(self.cpu.registers["ACC"])
            proc.runtime_cycles += 1

            # Update the task signature based on hardware rotation back-propagation
            proc.vector = np.copy(self.cpu.registers["ACC"])

            # Re-triage process to see if its vector state shifted or collapsed into a shortfall
            self.active_queue.remove(pid)
            self._triage_process(proc)

        print("--- End of Scheduler Cycle ---\n")

# =====================================================================
# OS KERNEL SANDBOX TEST RUN
# =====================================================================
if __name__ == "__main__":
    # Initialize our hardware stack
    hardware_core = TrinaryProcessor()
    kernel = TrinaryOS(hardware_core)

    # Define placeholder callback behaviors for testing
    def mock_ability_one(wavefront):
        print(f"  -> Ability One running on wave shape: {wavefront[:3]}...")

    def mock_ability_two(wavefront):
        print(f"  -> Ability Two executing stream transformation...")

    print("Initializing Core OS Capabilities...")

    # Process 1: Active, dynamic data wave
    kernel.register_ability(
        name="TelemetryWaveDecoder",
        wavefront_signature=np.array([1., 0., 1., 1., 0., -1., 0., 1., -1., 0., 0., 1., 1., 0.]),
        callback=mock_ability_one
    )

    # Process 2: Pure unmapped shortfall state (standby)
    kernel.register_ability(
        name="QuantumStreamListener",
        wavefront_signature=np.zeros(14),
        callback=mock_ability_two
    )

    # Process 3: Presumed dead-end/noise string (Early-Exit Trigger)
    kernel.register_ability(
        name="CorruptedBufferGhost",
        wavefront_signature=np.array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.]),
        callback=mock_ability_one
    )

    # Spin the operating system scheduler
    kernel.run_scheduler_cycle()

Initializing Core OS Capabilities...
KERNEL: Process 'TelemetryWaveDecoder' [PID 100] flagged as RELEVANT. Scheduled to Active Queue.
KERNEL: Process 'QuantumStreamListener' [PID 101] flagged as UNKNOWN. Parked in Shortfall Standby.
KERNEL: Process 'CorruptedBufferGhost' [PID 102] flagged as IRRELEVANT. Routed to Void Queue.

--- OS Scheduler Cycle Execution ---
Active Tasks: 1 | Shortfall Standby: 1 | Void Cleaned: 1
  [Purge Engine] Garbage collecting 1 dead-end threads to preserve registers.
  -> Ability One running on wave shape: [-0.  0. -0.]...
KERNEL: Process 'TelemetryWaveDecoder' [PID 100] flagged as UNKNOWN. Parked in Shortfall Standby.
--- End of Scheduler Cycle ---



In [4]:

import numpy as np
import time
from typing import Dict, List, Callable

# =====================================================================
# 1. TRINARY STATE DEFINITIONS
# =====================================================================
class TrinaryLogic:
    RELEVANT = 1      # Wave Crest (+1)   -> Active Value
    UNKNOWN = 0       # Ground Neutral (0) -> Structural Data Absence
    IRRELEVANT = -1   # Wave Trough (-1)   -> Path Void / Early Exit Trigger

# =====================================================================
# 2. HARDWARE EMULATOR MODULE: T-CPU
# =====================================================================
class TrinaryProcessor:
    """
    Emulates a 14-node Balanced Ternary Processor.
    Operates natively on vectors of (-1, 0, +1) using wave-domain operations.
    """
    def __init__(self, num_nodes: int = 14):
        self.num_nodes = num_nodes
        self.registers = {
            "R1": np.zeros(self.num_nodes),
            "R2": np.zeros(self.num_nodes),
            "ACC": np.zeros(self.num_nodes),
            "SR": 0
        }
        self.phi = 1.61803398875
        self.sieve_threshold = np.pi * self.phi * (1.0 / 3.0)
        self.lambda_max = 1.35

    def load_register(self, reg_name: str, wave_vector: np.ndarray):
        if len(wave_vector) != self.num_nodes:
            raise ValueError(f"Vector must precisely match the {self.num_nodes}-node fabric grid.")
        self.registers[reg_name] = np.copy(wave_vector)

    def execute_and(self, src_reg1: str, src_reg2: str, dest_reg: str):
        v1 = self.registers[src_reg1]
        v2 = self.registers[src_reg2]
        self.registers[dest_reg] = np.minimum(v1, v2)
        self._update_status_register(dest_reg)

    def execute_or(self, src_reg1: str, src_reg2: str, dest_reg: str):
        v1 = self.registers[src_reg1]
        v2 = self.registers[src_reg2]
        self.registers[dest_reg] = np.maximum(v1, v2)
        self._update_status_register(dest_reg)

    def execute_refract_rotation(self, src_reg: str, dest_reg: str):
        state = np.copy(self.registers[src_reg])
        rotation_matrix = np.roll(state, 1) * (1.0 / 3.0)
        new_state = (state * np.cos(self.sieve_threshold)) + (rotation_matrix * np.sin(self.sieve_threshold))
        self.registers[dest_reg] = np.clip(np.round(new_state), -1.0, 1.0)
        self._update_status_register(dest_reg)

    def _update_status_register(self, monitored_reg: str):
        target_vector = self.registers[monitored_reg]
        if np.min(target_vector) == TrinaryLogic.IRRELEVANT and np.max(target_vector) <= 0.0:
            self.registers["SR"] = int(TrinaryLogic.IRRELEVANT)
        elif np.all(target_vector == 0.0):
            self.registers["SR"] = int(TrinaryLogic.UNKNOWN)
        else:
            self.registers["SR"] = int(TrinaryLogic.RELEVANT)

# =====================================================================
# 3. OPERATING SYSTEM KERNEL MODULE: T-OS
# =====================================================================
class TrinaryProcess:
    """Represents a native task running on TrinaryOS."""
    def __init__(self, pid: int, name: str, execution_vector: np.ndarray, ability_callback: Callable):
        self.pid = pid
        self.name = name
        self.vector = np.copy(execution_vector)
        self.ability_callback = ability_callback
        self.state = TrinaryLogic.UNKNOWN
        self.runtime_cycles = 0

class TrinaryOS:
    """The core operating system kernel."""
    def __init__(self, hardware_cpu: TrinaryProcessor):
        self.cpu = hardware_cpu
        self.process_table: Dict[int, TrinaryProcess] = {}
        self.next_pid = 100
        self.active_queue: List[int] = []
        self.shortfall_queue: List[int] = []
        self.void_queue: List[int] = []

    def register_ability(self, name: str, wavefront_signature: np.ndarray, callback: Callable) -> int:
        pid = self.next_pid
        process = TrinaryProcess(pid, name, wavefront_signature, callback)
        self.process_table[pid] = process
        self.next_pid += 1
        self._triage_process(process)
        return pid

    def _triage_process(self, process: TrinaryProcess):
        min_state = np.min(process.vector)
        max_state = np.max(process.vector)

        if min_state == TrinaryLogic.IRRELEVANT and max_state <= 0.0:
            process.state = TrinaryLogic.IRRELEVANT
            if process.pid not in self.void_queue:
                self.void_queue.append(process.pid)
            print(f"KERNEL: Task '{process.name}' [PID {process.pid}] -> VOID Queue (Purge Target).")
        elif np.all(process.vector == 0.0):
            process.state = TrinaryLogic.UNKNOWN
            if process.pid not in self.shortfall_queue:
                self.shortfall_queue.append(process.pid)
            print(f"KERNEL: Task '{process.name}' [PID {process.pid}] -> SHORTFALL Standby Queue.")
        else:
            process.state = TrinaryLogic.RELEVANT
            if process.pid not in self.active_queue:
                self.active_queue.append(process.pid)
            print(f"KERNEL: Task '{process.name}' [PID {process.pid}] -> ACTIVE Processing Queue.")

    def run_scheduler_cycle(self):
        print(f"\n--- OS Clock Tick: Executing Trinary Scheduler ---")

        # 1. Early-Exit Purge
        if self.void_queue:
            print(f"  [Early-Exit Engine] Instantly clearing {len(self.void_queue)} irrelevant threads.")
            for pid in list(self.void_queue):
                del self.process_table[pid]
                self.void_queue.remove(pid)

        # 2. Hardware Register Processing
        for pid in list(self.active_queue):
            proc = self.process_table[pid]
            self.cpu.load_register("R1", proc.vector)
            self.cpu.execute_refract_rotation("R1", "ACC")

            # Execute ability logic block
            proc.ability_callback(self.cpu.registers["ACC"])
            proc.runtime_cycles += 1
            proc.vector = np.copy(self.cpu.registers["ACC"])

            self.active_queue.remove(pid)
            self._triage_process(proc)
        print("--- End of Scheduler Cycle ---\n")

# =====================================================================
# 4. EXECUTION ENGINE SANDBOX RUN
# =====================================================================
if __name__ == "__main__":
    t_cpu = TrinaryProcessor()
    t_os = TrinaryOS(t_cpu)

    def ability_demo(wavefront):
        print(f"  -> Executing capability logic on wavefront context: {wavefront[:4]}")

    print("Registering System Tasks...")
    t_os.register_ability("TelemetryDecoder", np.array([1., 0., 1., 1., 0., -1., 0., 1., -1., 0., 0., 1., 1., 0.]), ability_demo)
    t_os.register_ability("ShortfallListener", np.zeros(14), ability_demo)
    t_os.register_ability("NoiseGhost", np.array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.]), ability_demo)

    t_os.run_scheduler_cycle()

Registering System Tasks...
KERNEL: Task 'TelemetryDecoder' [PID 100] -> ACTIVE Processing Queue.
KERNEL: Task 'ShortfallListener' [PID 101] -> SHORTFALL Standby Queue.
KERNEL: Task 'NoiseGhost' [PID 102] -> VOID Queue (Purge Target).

--- OS Clock Tick: Executing Trinary Scheduler ---
  [Early-Exit Engine] Instantly clearing 1 irrelevant threads.
  -> Executing capability logic on wavefront context: [-0.  0. -0.  0.]
KERNEL: Task 'TelemetryDecoder' [PID 100] -> SHORTFALL Standby Queue.
--- End of Scheduler Cycle ---

